In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import *

_order_status = spark.conf.get('customer.orderStatus',"NA")

@dp.table(name="debug_config")
def debug_config():
    return spark.sql(f"SELECT '{_order_status}' AS order_status")

In [0]:
#rules (warn, drop, fail)
__order_rules = {
    "valid order status":"o_orderstatus in ('O','F','P')",
    "valid order price":"o_totalprice>0"
}

__customer_rules = {
    "valid market segment":"c_mktsegment is not null"
}

In [0]:


# @dp.table(
#     name="customers",
#     table_properties={"quality":"bronze"}

# )
# def customers_dlt():
#     return spark.read.table("dev.dlt.customers_dlt")

In [0]:
# creating a materialized view for customers
@dp.view(
    comment="customer bronze view"
)
@dp.expect_all_or_drop(__customer_rules)
def customers_bronze():
    return spark.readStream.table("dev.dlt.customers_dlt")

In [0]:
dp.create_streaming_table('customer_scd1_bronze')

#scd 1 customer
dp.apply_changes(
    target="customer_scd1_bronze",
    source="customers_bronze",
    keys=["c_custkey"],
    stored_as_scd_type=1,
    apply_as_deletes=expr("_src_action=='D'"),
    apply_as_truncates=expr("_src_action=='T'"),
    sequence_by="_src_insert_dt"
)


In [0]:
dp.create_streaming_table('customer_scd2_bronze')

#scd 2 customer
dp.apply_changes(
    target="customer_scd2_bronze",
    source="customers_bronze",
    keys=["c_custkey"],
    stored_as_scd_type=2,
    except_column_list=["_src_insert_dt","_src_action"],
    sequence_by="_src_insert_dt"
)


In [0]:
@dp.table(
    name="orders",
    table_properties={"quality":"bronze"}

)
@dp.expect_all_or_drop(__order_rules)
def customers_dlt():
    return spark.readStream.table("dev.dlt.orders_dlt")

In [0]:
@dp.table(
    name="orders_autoloader_bronze",
    table_properties={"quality":"bronze"}

)
def fn():
    df = (
        spark
        .readStream
        .format('cloudFiles')
        .option('cloudFiles.schemaHints',"o_orderkey long,o_custkey long,o_orderstatus string,o_totalprice decimal(18,2),o_orderdate date,o_orderpriority string,o_clerk string,o_shippriority integer,o_comment string")
        .option('cloudFiles.schemaLocation',"/Volumes/dev/dlt/dlt_volume/auto_loader/schemas/1/")
        .option('cloudFiles.format',"CSV")
        .option('pathGlobfilter',"*.csv")
        .option('cloudFiles.schemaEvolutionMode',"None")
        .load('/Volumes/dev/dlt/dlt_volume/files/')
    )
    return df

In [0]:
dp.create_streaming_table(
    name="orders_union_bronze"
)

# append orders flow
@dp.append_flow(
    target='orders_union_bronze'
)
def orders_delta_append():
    df = spark.readStream.table('orders')
    return df


# append flow
@dp.append_flow(
    target='orders_union_bronze'
)
def orders_autoloader_append():
    df = spark.readStream.table('orders_autoloader_bronze')
    return df

In [0]:
@dp.view(
    name='customer_orders'
)
def fn_customer_orders():
    df_c = spark.read.table("customer_scd2_bronze").where('__END_AT is null')
    df_o = spark.read.table('orders_union_bronze')
    df_join = df_c.join(df_o,how="left_outer",on=df_c.c_custkey==df_o.o_custkey)
    return df_join

In [0]:
from pyspark.sql.functions import * 

@dp.table(
    name='joined_silver',
    table_properties={
        "quality":"silver"}
)
def fn_customer_orders():
    df = spark.read.table('customer_orders').withColumn("_insertDate",current_timestamp())
    return df

In [0]:
from pyspark.sql.functions import * 

@dp.table(
    table_properties={
        "quality":"gold"}
)
def orders_aggregate_gold():
    df = spark.read.table('joined_silver')
    df_final = df.groupBy('c_mktsegment').agg(count('o_orderkey').alias('count_of_orders'),sum('o_totalprice').alias('sum_totalprice')).withColumn('_insertDate',current_timestamp())
    return df_final

In [0]:
for _status in _order_status.split(","):
    display(spark.sql(f"SELECT '{_order_status}' AS order_status"))
    from pyspark.sql.functions import * 

    @dp.table(
        table_properties={
            "quality":"gold"},
        name=f"orders_agg_{_status}_gold"
    )
    def fn():
        df = spark.read.table('joined_silver')
        df_final = df.where(f"o_orderstatus=='{_status}'").groupBy('c_mktsegment').agg(count('o_orderkey').alias('count_of_orders'),sum('o_totalprice').alias('sum_totalprice')).withColumn('_insertDate',current_timestamp())
        return df_final

